# Tutorial 08 — Governed execution: local lab

This is a **guided lab**, not a dump of printouts. Each block has a short **story** (why the layer
exists), a **knob** you can edit (`USER_*`, overlays, prompts), and **stdout** you read like an
operator would read logs and policy traces.

**Optional live contrasts** (API key, governed vs raw SDK) are in
**`tutorial_09_governed_execution_live.ipynb`** — run this notebook first.

**What you will understand**

1. **Ingress (pre-model)** — text enters the system; gates can **deny / escalate** before any model
   spend. You configure patterns and profiles like tenant overlay keys.
2. **Tool policy (risk + tenant overlay)** — every tool intent passes **policy** (`before_tool_call`).
   **Allow** lets execution continue; **deny / escalate** return structured **blocked** envelopes instead
   of running your Python handler.
3. **Execution mode** — for **low-risk, non-state-changing** work, the stack may route **provider-native**
   (model/SDK path). For **high-risk or state-changing** work, eXo-brain **forces deterministic** execution:
   your registered **handler** runs inside `DeterministicToolExecutor`, so the model only sees **typed
   `ToolResult`**, not raw side effects. That is how governance stays **accurate and auditable**.

**How to run it**

- Run **top to bottom** the first time so `policy_overlay`, `registry`, `executor`, and `chain` exist.
- **Parts 1–7** need **no API key** (local policy, ingress, stub orchestrator, and `planned_tool_call`).
- Continue with **`tutorial_09`** for optional live OpenAI contrasts.

**Requires:** `pip install -r requirements.txt` (all four PyPI wheels: `exo-brain-core-contracts`,
`exo-brain-adapter-sdk`, `exo-adapter-echo`, `exo-adapter-openai`).

**Further reading:** `docs/architecture/governed-execution-pipeline.md` (ordering of ingress, orchestrator,
policy, deterministic tools on the full API path).

## For non-technical readers

You do **not** need to read Python to get value from this lab. Use this box as your **executive path**,
then skim each Part’s **Story** heading (skip code if you prefer).

### The problem in one sentence

Teams want helpful AI — but **not** at the price of leaking secrets, triggering dangerous actions, or
racking up model and tool spend with **no trace** of who allowed what.

### What you gain (business language)

| You gain | What it feels like day to day |
|----------|------------------------------|
| **Safety** | Risky or sensitive input can be **stopped or sent for review** *before* the model runs. |
| **Control** | Rules decide **what may run** — not vibes from the model. |
| **Predictability** | Important outcomes can follow **repeatable** logic you can test, not one-off guesses. |
| **Proof** | Allow/deny decisions carry **reasons** you can show support, security, or auditors. |
| **Cost discipline** | Problems caught **early** mean fewer wasted tokens and tool calls. |

### The “three numbers” proof (Part 4 + optional tutorial_09)

Imagine asking for **11 + 33**. A model can say **44** from memory. In this lab, the real answer is
**11 + 33 + a secret third addend** only your server knows — plus a **proof code** the model cannot invent.
Part **4** prints **`[PASS] Part 4 local proof`** when handler JSON matches your kernel. **tutorial_09** §3
prints **`§3 VERIFICATION (governed): PASS`** when **`planned_tool_call`** shows **`tool_progress` completed**
*and* the assistant cites kernel **sum** + **proof_token** (not mental math).

### What you will *see* when someone runs the cells

- **Healthy path:** words like *allow*, *completed*, or a clear numeric result from a safe tool.
- **Governance doing its job:** *deny*, *blocked*, *escalate*, or a short **reason code** — that is the
  product **protecting you**, not a random error.

### Two ways to use this notebook

1. **Executive path (~3 minutes):** this box → **Map** table below → each Part’s **Story** only.
2. **Hands-on path (~15 minutes):** run **top to bottom**; tweak **Your task** knobs and watch stdout.
   Continue with **`tutorial_09`** for optional live contrasts (OpenAI API key).

### Jargon cheat sheet (plain words ↔ what engineers say)

| Engineers say | You can picture |
|-----------------|------------------|
| Ingress | The **door** that reads the message **before** the AI. |
| Policy / risk gates | **Automatic rules** for safe vs risky actions. |
| Deterministic tools | The work ran in **our** code path so the **answer is checkable**. |
| Tenant overlay | **Extra rules for one customer** without changing everyone else’s defaults. |

With an API key, **`tutorial_09`** runs short **governed vs raw** comparisons (ingress, blocked tool, proof math,
optional calc). **`§N VERIFICATION (governed): PASS/FAIL`** lines report each section; **§2–§4** use
**`planned_tool_call`** (same as Part 7) so governed proofs do **not** depend on the model choosing tools.
Use **`NB_LIVE_*`** env flags to skip sections and save tokens.

## Beginner checklist — read this once

1. **Run cells from the top** the first time (bootstrap → Part 1 → …). Later you can jump back to any
   **Part** after the variables it needs exist (`policy_risk`, `policy_overlay`, `registry`, `executor`,
   `chain`).
2. Each **Part** has three cues: **Story** (why), **Your task** (what to edit), **Reading stdout** (what
   good looks like). If stdout confuses you, re-read **Story** for that part only.
3. **“With vs without”** appears in several code cells: the notebook prints **two** behaviours side by
   side (strict vs relaxed policy, overlay on vs off, direct Python vs governed executor). That is
   intentional so you see *what the framework adds*.
4. **No API key required here.** Optional live contrasts are in **`tutorial_09`**.

**Typical first run:** ~10–20 minutes without an API key (includes **Checkpoint** + divide-by-zero demo).
Add **`tutorial_09`** for live contrasts (~5–10 minutes with a key).

## Map — where you are in the stack

| Stage | You configure (examples) | This notebook |
|-------|--------------------------|----------------|
| Ingress | `INGRESS_OVERLAY`, profiles, custom rules | **Part 6** |
| Tool policy | `USER_RISK`, `USER_OVERLAY`, tenant id on `ToolCallContext` | **Parts 1–3** |
| Deterministic tools | `USER_TOOLS`; **`safe_add_proven`** (3-operand sum + proof) and **`calculate_result`** | **Part 4** |
| Execution mode | Capability map + policy `enforced_mode` | **Part 5** |
| Orchestrator stream | `planned_tool_call` (stub) | **Part 7** |
| Live contrasts (optional) | `OPENAI_API_KEY`, `NB_LIVE_*` flags | **`tutorial_09`** |

**Integrator note:** Calling `Orchestrator.run_turn` directly **skips** HTTP-only steps (some entitlements,
budgets). Here we **explicitly** run ingress before live contrasts (**`tutorial_09`**) to mirror the *spirit* of the pipeline doc;
production traffic should still go through **`src/api/routers/turns.py`** (SSE/WebSocket turn execution)
when you integrate — that router applies the full ingress + orchestration stack for real tenants.

**CI:** On pull requests touching `notebooks/**`, CI executes **`tutorial_08`** with **`nbconvert`**
(no API key). If execution fails, fix **`notebooks/build_tutorials.py`** and regenerate notebooks.

## Orientation — table of contents and pipeline (read once)

**Rough time per part** (reading Story + running code):

| Part | Topic | ~Time |
|------|--------|------|
| Bootstrap | paths, `.env` | 1 min |
| 1–2 | Risk gates + synthetic probes | 2–4 min |
| 3 | Tenant overlay (**DENY** vs **ESCALATE** cue) | 2 min |
| 4 | Registry tools + `run_tool` + divide-by-zero error | 4–6 min |
| 5 | Execution mode sweep | 2 min |
| 6 | Ingress gate chain | 3 min |
| **Checkpoint** | verify globals before orchestrator | 30 s |
| 7 | Stub orchestrator stream | 2 min |

**Optional:** **`tutorial_09_governed_execution_live.ipynb`** — live contrasts (~5–10 min with API key).

**Happy-path pipeline** (this notebook mirrors the middle layers; HTTP adds more gates upstream):

```mermaid
flowchart LR
  U[User text] --> I[Ingress gate chain]
  I -->|ALLOW| O[Orchestrator.run_turn]
  I -->|DENY / ESCALATE| X[Stop before model]
  O --> R[Runtime adapter]
  R --> P[Policy before_tool_call]
  P -->|DENY / ESCALATE| B[Blocked envelope]
  P -->|ALLOW| E[DeterministicToolExecutor]
  E --> H[Your Python handler]
  H --> T[ToolResult to model]
```

In **Cursor / VS Code** and on **GitHub**, the diagram renders from the `mermaid` fence. Plain Jupyter may
show the fence as text unless a Mermaid extension is installed — the **ASCII** takeaway is still:
**ingress → orchestrator → policy → executor → handler**.

## Story — Why “deterministic tools” help the agent answer correctly

The model proposes **names and arguments** for tools. **Governance** decides whether that proposal
may run, and **how** it runs:

- **Deterministic path:** your Python **handler** runs in the executor. The model receives a
  **`ToolResult`** with stable fields (`status`, `error`, audit correlation). Side effects match what
  you coded — not what the SDK guessed.
- **Provider-native path:** the adapter may let the Agents SDK continue the tool loop. That is useful
  for low-risk flows when policy and capability maps agree — but it is **not** where you want silent
  writes or high-risk actions.

So: **deterministic tools do not “make the LLM smarter”** — they **bound** what actually happened so
the **next** model token is grounded in **your** truth, which is what operators mean by a trustworthy
agent response.

In [1]:
import asyncio
import traceback
import importlib
import importlib.util

import pathlib
import sys

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))
try:
    from dotenv import load_dotenv
    load_dotenv(_root / ".env", override=False)
except ImportError:
    pass

print("repo root:", _root.name)

_ADAPTER_WHEELS = (
    ("exo-brain-core-contracts", "exo_brain_core_contracts"),
    ("exo-brain-adapter-sdk", "exo_brain_adapter_sdk"),
    ("exo-adapter-echo", "exo_adapter_echo"),
    ("exo-adapter-openai", "exo_adapter_openai"),
)

for dist, module_name in _ADAPTER_WHEELS:
    if importlib.util.find_spec(module_name) is None:
        print(f"warn: {dist} not installed — pip install -r requirements.txt")
        continue
    mod = importlib.import_module(module_name)
    mod_file = (mod.__file__ or "").replace("\\", "/")
    if "site-packages" not in mod_file and "dist-packages" not in mod_file:
        raise RuntimeError(f"{dist} must be a PyPI wheel in site-packages, got {mod.__file__}")
    if "/eXo_adapters/" in mod_file:
        raise RuntimeError(
            f"{dist} must not load from eXo_adapters checkout — "
            f"pip install -r requirements.txt: {mod.__file__}"
        )
    print(f"{dist}: <site-packages>/{module_name}")

from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter

assert OpenAIAgentsRuntimeAdapter.__module__.startswith("exo_adapter_openai."), (
    "OpenAIAgentsRuntimeAdapter must come from exo-adapter-openai (PyPI); "
    "reinstall: pip install -r requirements.txt"
)
print("OpenAIAgentsRuntimeAdapter module:", OpenAIAgentsRuntimeAdapter.__module__)

repo root: eXo-brain
exo-brain-core-contracts: <site-packages>/exo_brain_core_contracts
exo-brain-adapter-sdk: <site-packages>/exo_brain_adapter_sdk
exo-adapter-echo: <site-packages>/exo_adapter_echo
exo-adapter-openai: <site-packages>/exo_adapter_openai
OpenAIAgentsRuntimeAdapter module: exo_adapter_openai.runtime


## Part 1 — Risk gate knobs (`RiskGateConfig`)

**Story.** Before any runtime adapter runs, product policy usually includes **tier and tool rules**:
which tiers must never execute unattended, which tools always need review, and whether **any**
state-changing call should escalate. `RiskGateConfig` is the declarative bundle for those rules.

**Your task.** Edit **`USER_RISK`** (string tier names and exact tool names), then run the code cell.

**Reading stdout.** This cell only confirms the config object was built. **Part 2** prints one line per
synthetic intent: `decision` (`allow` / `deny` / `escalate`), `reason_code`, `review_required`,
`enforced_mode`.

- `deny_risk_tiers` / `escalate_risk_tiers`: e.g. `"high"`, `"critical"`.
- `deny_tools` / `escalate_tools`: exact registry tool names.
- `escalate_state_changing`: when true, any `is_state_changing=True` intent escalates.

In [2]:
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.policies.risk_gates import RiskGateConfig
from src.schemas.tool_io import PolicyAction, RiskTier, ToolCallContext

# ── edit below ─────────────────────────────────────────────────────────────
USER_RISK = {
    "deny_risk_tiers": [],           # e.g. ["critical"]
    "escalate_risk_tiers": ["high"], # demo: HIGH -> ESCALATE
    "deny_tools": [],
    "escalate_tools": [],
    "escalate_state_changing": False,
    "review_channel": "notebook-review",
}


def _tiers(keys: list[str]) -> set[RiskTier]:
    out: set[RiskTier] = set()
    for k in keys:
        try:
            out.add(RiskTier(str(k)))
        except ValueError:
            print("skip unknown RiskTier:", k)
    return out


risk_cfg = RiskGateConfig(
    deny_risk_tiers=_tiers(USER_RISK["deny_risk_tiers"]),
    escalate_risk_tiers=_tiers(USER_RISK["escalate_risk_tiers"]),
    deny_tools=set(USER_RISK["deny_tools"]),
    escalate_tools=set(USER_RISK["escalate_tools"]),
    escalate_state_changing=bool(USER_RISK["escalate_state_changing"]),
    review_channel=str(USER_RISK["review_channel"]),
)
policy_risk = DeterministicFirstPolicyMiddleware(risk_gate_config=risk_cfg)
assert USER_RISK["escalate_risk_tiers"] == ["high"]
assert risk_cfg.review_channel == "notebook-review"
print("RiskGateConfig ready:", USER_RISK)

RiskGateConfig ready: {'deny_risk_tiers': [], 'escalate_risk_tiers': ['high'], 'deny_tools': [], 'escalate_tools': [], 'escalate_state_changing': False, 'review_channel': 'notebook-review'}


## Part 2 — Probe `before_tool_call` (synthetic tool intents)

**Story.** `PolicyMiddleware.before_tool_call` is the **same** function the orchestrator invokes when
a runtime adapter emits a **tool intent**. This block lets you experiment **without** a model: each
scenario is a hand-built `ToolCallContext`.

**Your task.** Edit **`SCENARIOS`** (tool name, risk tier string, state-changing flag). Re-run.

**Reading stdout.** Each scenario prints **twice**: first with your **Part 1** rules (`policy_risk`),
then with a **relaxed** risk config (`policy_permissive`) so you can see the same tool intent **with**
and **without** tier escalation. For the defaults, **s2** (`delete_row`, HIGH, state-changing) should
move from **escalate** → **allow** (still **deterministic** at execution time — see Part 5).

**Contrast you should internalize:** governance is not “off vs on” — it is **which rules fire** for the
same call shape.

In [3]:
from src.schemas.tool_io import PolicyAction, PolicyDecision, RiskTier, ToolCallContext, ToolExecutionMode

SCENARIOS = [
    {"call_id": "s1", "tool": "read_db", "risk": "low", "state": False},
    {"call_id": "s2", "tool": "delete_row", "risk": "high", "state": True},
    {"call_id": "s3", "tool": "admin_reset", "risk": "medium", "state": True},
]


def _ctx(entry: dict) -> ToolCallContext:
    return ToolCallContext(
        schema_version="1.0",
        call_id=entry["call_id"],
        session_id="nb_sess",
        run_id="nb_run",
        job_id="nb_job",
        task_id="nb_task",
        agent_id="nb_agent",
        provider_id="demo",
        tool_name=entry["tool"],
        arguments={},
        tenant_id="tenant_nb",
        risk_tier=RiskTier(str(entry["risk"])),
        is_state_changing=bool(entry["state"]),
    )


policy_permissive = DeterministicFirstPolicyMiddleware(risk_gate_config=RiskGateConfig())

risk_results: dict[str, PolicyDecision] = {}
print("--- with YOUR Part 1 rules (USER_RISK) ---")
for row in SCENARIOS:
    d = policy_risk.before_tool_call(_ctx(row))
    risk_results[row["call_id"]] = d
    print(
        row["call_id"],
        d.decision.value,
        d.reason_code,
        "review=" + str(d.review_required),
        "enforced_mode=" + str(d.enforced_mode),
    )

print("\n--- contrast: relaxed risk gates (empty RiskGateConfig, same SCENARIOS) ---")
for row in SCENARIOS:
    d = policy_permissive.before_tool_call(_ctx(row))
    print(
        row["call_id"],
        d.decision.value,
        d.reason_code,
        "review=" + str(d.review_required),
        "enforced_mode=" + str(d.enforced_mode),
    )

assert risk_results["s1"].decision == PolicyAction.ALLOW
assert risk_results["s1"].reason_code == "LOW_RISK_ALLOWED"
assert risk_results["s1"].enforced_mode is None

assert risk_results["s2"].decision == PolicyAction.ESCALATE
assert risk_results["s2"].reason_code == "RISK_TIER_REQUIRES_REVIEW"
assert risk_results["s2"].review_required is True
assert risk_results["s2"].enforced_mode == ToolExecutionMode.DETERMINISTIC

assert risk_results["s3"].decision == PolicyAction.ALLOW
assert risk_results["s3"].reason_code == "RISK_WRITE_REQUIRES_DETERMINISTIC"
assert risk_results["s3"].enforced_mode == ToolExecutionMode.DETERMINISTIC

relaxed_s2 = policy_permissive.before_tool_call(_ctx(SCENARIOS[1]))
assert relaxed_s2.decision == PolicyAction.ALLOW
assert relaxed_s2.reason_code == "RISK_WRITE_REQUIRES_DETERMINISTIC"
assert relaxed_s2.enforced_mode == ToolExecutionMode.DETERMINISTIC

--- with YOUR Part 1 rules (USER_RISK) ---
s1 allow LOW_RISK_ALLOWED review=False enforced_mode=None
s2 escalate RISK_TIER_REQUIRES_REVIEW review=True enforced_mode=ToolExecutionMode.DETERMINISTIC
s3 allow RISK_WRITE_REQUIRES_DETERMINISTIC review=False enforced_mode=ToolExecutionMode.DETERMINISTIC

--- contrast: relaxed risk gates (empty RiskGateConfig, same SCENARIOS) ---
s1 allow LOW_RISK_ALLOWED review=False enforced_mode=None
s2 allow RISK_WRITE_REQUIRES_DETERMINISTIC review=False enforced_mode=ToolExecutionMode.DETERMINISTIC
s3 allow RISK_WRITE_REQUIRES_DETERMINISTIC review=False enforced_mode=ToolExecutionMode.DETERMINISTIC


## Part 3 — Tenant policy overlay (same risk engine, per-tenant)

**Story.** Global defaults rarely survive multi-tenant reality. `TenantPolicyOverlayStore` merges
**per-tenant** overlay keys onto the same risk gate engine — think “this customer blocks `admin_reset`
even if global policy only escalates HIGH.”

**Your task.** Edit **`USER_OVERLAY`** for tenant `tenant_nb`, then run. Keys mirror overlay fields read
by `RiskGatePolicy` (see `src/policies/risk_gates.py`).

**Reading stdout.** The cell prints **two** decisions for the **same** `ToolCallContext`: first
**without** a tenant overlay on policy (global risk rules only), then **with** `tenant_nb` overlay
(`admin_reset` on the deny list). Beginners should see `allow` flip to **`deny`** only when the overlay
is applied — that is what “per-tenant guard rail” means in code.

**DENY vs ESCALATE (same cell):** the second block probes a **HIGH** risk, state-changing intent. With
**Part 1** defaults (`escalate_risk_tiers` includes **high**), policy returns **`escalate`** — *review
queue semantics*, not a hard block. Compare that feeling to **`deny`** on `admin_reset` above.

**Try this:** remove `"admin_reset"` from `deny_tools`, re-run, and watch the second line follow the first.

In [4]:
from src.tenancy.policy_overlay import TenantPolicyOverlayStore
from src.schemas.tool_io import PolicyAction, RiskTier, ToolCallContext, ToolExecutionMode

USER_OVERLAY = {
    "deny_tools": ["admin_reset"],
    "escalate_state_changing": False,
    "review_channel": "tenant-security",
}

overlays = TenantPolicyOverlayStore()
overlays.set_overlay("tenant_nb", USER_OVERLAY)
policy_overlay = DeterministicFirstPolicyMiddleware(
    risk_gate_config=risk_cfg,
    tenant_policy_overlays=overlays,
)

probe = ToolCallContext(
    schema_version="1.0",
    call_id="ov1",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="admin_reset",
    arguments={},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.MEDIUM,
    is_state_changing=False,
)
policy_global_only = DeterministicFirstPolicyMiddleware(risk_gate_config=risk_cfg)
dec_global = policy_global_only.before_tool_call(probe)
print("without tenant overlay (global risk only):", dec_global.decision.value, dec_global.reason_code)

dec = policy_overlay.before_tool_call(probe)
dec_msg = dec.message
assert dec_msg is not None
print("with tenant_nb overlay (USER_OVERLAY):    ", dec.decision.value, dec.reason_code, dec_msg[:120])

probe_escalate = ToolCallContext(
    schema_version="1.0",
    call_id="ov_esc",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="delete_row",
    arguments={},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.HIGH,
    is_state_changing=True,
)
esc = policy_overlay.before_tool_call(probe_escalate)
print(
    "HIGH+state delete_row (Part 1 escalate_risk_tiers):",
    esc.decision.value,
    esc.reason_code,
    "review_required=" + str(getattr(esc, "review_required", False)),
)

assert dec_global.decision == PolicyAction.ALLOW
assert dec_global.reason_code == "LOW_RISK_ALLOWED"

assert dec.decision == PolicyAction.DENY
assert dec.reason_code == "TOOL_DENIED"
assert "blocked by policy" in dec_msg.lower()

assert esc.decision == PolicyAction.ESCALATE
assert esc.reason_code == "RISK_TIER_REQUIRES_REVIEW"
assert esc.review_required is True
assert esc.enforced_mode == ToolExecutionMode.DETERMINISTIC

assert overlays.get_overlay("tenant_nb") == USER_OVERLAY

without tenant overlay (global risk only): allow LOW_RISK_ALLOWED
with tenant_nb overlay (USER_OVERLAY):     deny TOOL_DENIED Tool is blocked by policy configuration.
HIGH+state delete_row (Part 1 escalate_risk_tiers): escalate RISK_TIER_REQUIRES_REVIEW review_required=True


## Part 4 — Deterministic tools you define (handlers + policy + executor)

**Story.** The **deterministic tool runtime** is the contract boundary: the model never executes your
handler. `DeterministicToolExecutor` validates, applies policy again (defense in depth), runs **`fn`**
in-process, and returns a **`ToolResult`**. That is the path you rely on for **money-moving**,
**data-changing**, or **high-risk** operations.

**Your task.** Edit **`USER_TOOLS`**: each item is `{"name", "risk", "state", "fn"}` with **`fn`** a
plain Python callable. Optional keys: **`description`**, **`parameters_schema`** (JSON Schema for the
OpenAI tool surface — used when you go live in **tutorial_09**). Re-run `run_tool(...)` at the bottom or add
your own.

**Reading stdout.** `before:` shows policy on the intent. `execute status:` shows executor reality
(`success` vs `blocked`). `mode_used` echoes which execution mode was recorded on the envelope — in
this notebook it stays **`deterministic`** whenever policy blocks or the call is high-impact.

**`calculate_result` (Tutorial 02 parity):** same handler shape as **`tutorial_02_openai_adapter`** —
`operation` (`add` / `subtract` / `multiply` / `divide`) plus **`operand1`** / **`operand2`**. Here it
is registered only in **`ToolRegistry`** (no `@function_tool` in this cell): **`OpenAIAgentsRuntimeAdapter`**
builds SDK tools from the registry (**`build_agent_tools`**), so you see one way production wiring reuses
the same contract as the adapter tutorial.

**`safe_add_proven` (enterprise proof tool):** registered as **MEDIUM + state-changing** so production-style
orchestration prefers the **deterministic executor** (same trust boundary as audited financial tools).
The model supplies **`a`** and **`b`** only; the handler adds **`random_operand`** from per-kernel
**`NB_FORMULA_SECRET`** (never in the user prompt). **`sum = a + b + random_operand`**.

**Acceptance criteria (Part 4 stdout — your kernel’s numbers will differ):**

| Check | Pass signal | Fail signal |
|-------|-------------|-------------|
| Hidden addend | `random_operand` printed (e.g. **4746**) | Only **a+b** appears |
| Governed sum | JSON **`sum`** = a+b+random (e.g. **4751** for a=2,b=3) | **`sum`** equals plain **5** |
| Proof | **`proof_token`** matches **`NB_FORMULA_SECRET`** | Missing or invented token |
| Path | `run_tool` → **`mode_used: DETERMINISTIC`** | Direct `safe_add_proven(...)` only (no policy shell) |

**tutorial_09 §3 (optional, API key):** replays **11+33** live and prints **`[PASS]` / `[FAIL]`** lines — pass
requires **`safe_add_proven` completed** on the orchestrator path *then* **your** kernel **`sum`** and
**`proof_token`** in the reply (not **44** from mental math or parroting the operator baseline).

**Structured errors:** one **`calculate_result`** call **divides by zero** so you see a deterministic
**`ToolResult`** in **`error`** shape (handler raises; executor wraps — same story as Tutorial 02’s
division demo).

**Contrast at the bottom of the cell:** you will also see **`safe_add` called as plain Python** (no
policy, no metrics). That number is *not* what a production agent path would use — it only shows what
“no governance shell” looks like next to the **same** operation through **`run_tool`**.

In [5]:
import secrets

from src.observability.metrics import RuntimeMetrics
from src.schemas.tool_io import ToolExecutionMode, ToolResult, ToolStatus
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolDescriptor, ToolRegistry


def safe_add(a: int, b: int) -> int:
    return a + b


def risky_echo(msg: str) -> str:
    return msg.upper()


def admin_reset() -> str:
    # Demo handler; tenant overlay denies this tool name before it runs in governed paths.
    return "admin-reset-handler-ran"


# Unpredictable per kernel: third addend + proof_token — not visible in the user prompt.
NB_FORMULA_SECRET = secrets.token_hex(8)


def _nb_random_operand() -> int:
    """Stable random addend for this kernel (100..8999); only the handler knows it."""
    return 100 + (int(NB_FORMULA_SECRET[:8], 16) % 8900)


def safe_add_proven(a: int, b: int) -> dict[str, object]:
    a_i, b_i = int(a), int(b)
    random_operand = _nb_random_operand()
    total = a_i + b_i + random_operand
    return {
        "operand_a": a_i,
        "operand_b": b_i,
        "random_operand": random_operand,
        "sum": total,
        "proof_token": NB_FORMULA_SECRET,
        "formula": f"{a_i}+{b_i}+{random_operand}=={total}",
    }


def _nb_calculate_result(operation: str, operand1: float, operand2: float) -> dict[str, object]:
    """Same arithmetic contract as tutorial_02 (registry handler; policy + executor wrap it here)."""
    op = str(operation).strip().lower()
    if op == "add":
        value = float(operand1) + float(operand2)
    elif op == "subtract":
        value = float(operand1) - float(operand2)
    elif op == "multiply":
        value = float(operand1) * float(operand2)
    elif op == "divide":
        if float(operand2) == 0:
            raise ValueError("division by zero is not allowed")
        value = float(operand1) / float(operand2)
    else:
        raise ValueError(f"unknown operation: {operation!r}")
    return {
        "operation": op,
        "operand1": float(operand1),
        "operand2": float(operand2),
        "result": value,
    }


_CALCULATE_RESULT_SCHEMA: dict[str, object] = {
    "type": "object",
    "properties": {
        "operation": {
            "type": "string",
            "description": "One of: add, subtract, multiply, divide",
        },
        "operand1": {"type": "number"},
        "operand2": {"type": "number"},
    },
    "required": ["operation", "operand1", "operand2"],
}

_SAFE_ADD_PROVEN_SCHEMA: dict[str, object] = {
    "type": "object",
    "properties": {
        "a": {"type": "integer", "description": "First addend (visible to the model)."},
        "b": {"type": "integer", "description": "Second addend (visible to the model)."},
    },
    "required": ["a", "b"],
}


def _nb_print_proof_reference(a: int, b: int, *, title: str) -> tuple[int, int]:
    """Print kernel-only operands for demos; returns (random_operand, governed_sum)."""
    r = _nb_random_operand()
    governed = a + b + r
    plain = a + b
    print(title)
    print(f"  random_operand (handler-only): {r}")
    print(f"  governed sum {a}+{b}+{r} => {governed}  |  plain {a}+{b} => {plain} (wrong without tool)")
    print(f"  proof_token (this kernel): {NB_FORMULA_SECRET}")
    return r, governed


USER_TOOLS = [
    {"name": "safe_add", "risk": RiskTier.LOW, "state": False, "fn": safe_add},
    {
        "name": "safe_add_proven",
        "risk": RiskTier.MEDIUM,
        "state": True,
        "fn": safe_add_proven,
        "description": (
            "Adds a and b plus a hidden per-tenant random_operand; returns sum, formula, and proof_token."
        ),
        "parameters_schema": _SAFE_ADD_PROVEN_SCHEMA,
    },
    {
        "name": "calculate_result",
        "risk": RiskTier.LOW,
        "state": False,
        "fn": _nb_calculate_result,
        "description": "Basic arithmetic: add, subtract, multiply, or divide two operands.",
        "parameters_schema": _CALCULATE_RESULT_SCHEMA,
    },
    {"name": "risky_echo", "risk": RiskTier.HIGH, "state": True, "fn": risky_echo},
    {"name": "admin_reset", "risk": RiskTier.MEDIUM, "state": True, "fn": admin_reset},
]

registry = ToolRegistry()
for spec in USER_TOOLS:
    registry.register(
        ToolDescriptor(
            name=spec["name"],
            handler=spec["fn"],
            risk_tier=spec["risk"],
            is_state_changing=spec["state"],
            description=str(spec.get("description", "")),
            parameters_schema=dict(spec["parameters_schema"]) if spec.get("parameters_schema") else {},
        )
    )

metrics = RuntimeMetrics()
executor = DeterministicToolExecutor(
    registry=registry,
    policy=policy_overlay,
    metrics=metrics,
)


def run_tool(
    name: str,
    args: dict,
    call_id: str,
    *,
    risk_tier: RiskTier = RiskTier.LOW,
    is_state_changing: bool = False,
) -> ToolResult:
    call = ToolCallContext(
        schema_version="1.0",
        call_id=call_id,
        session_id="nb_sess",
        run_id="nb_run",
        job_id="nb_job",
        task_id="nb_task",
        agent_id="nb_agent",
        provider_id="demo",
        tool_name=name,
        arguments=args,
        tenant_id="tenant_nb",
        risk_tier=risk_tier,
        is_state_changing=is_state_changing,
    )
    pre = policy_overlay.before_tool_call(call)
    print("before:", pre.decision.value, pre.reason_code)
    out = executor.execute(call)
    print("execute status:", out.status.value)
    err = out.error
    err_code = getattr(err, "code", None) if err is not None else None
    err_msg = getattr(err, "message", "") if err is not None else ""
    if err_msg is None:
        err_msg = ""
    print("  error:", err_code, str(err_msg)[:200])
    print("  mode_used:", out.execution.mode_used)
    return out


r_add = run_tool("safe_add", {"a": 2, "b": 3}, "tc_add", risk_tier=RiskTier.LOW, is_state_changing=False)
assert r_add.status == ToolStatus.SUCCESS
assert r_add.result == {"value": 5}
assert r_add.execution.mode_used == ToolExecutionMode.DETERMINISTIC

_demo_r, _demo_sum = _nb_print_proof_reference(
    2,
    3,
    title="-- safe_add_proven: enterprise proof (sum = a + b + kernel random_operand) --",
)
r_prov = run_tool(
    "safe_add_proven",
    {"a": 2, "b": 3},
    "tc_prov",
    risk_tier=RiskTier.MEDIUM,
    is_state_changing=True,
)
assert r_prov.status == ToolStatus.SUCCESS
assert r_prov.result is not None
payload = r_prov.result["value"]
print("  safe_add_proven JSON:", payload)
assert payload["operand_a"] == 2
assert payload["operand_b"] == 3
assert payload["random_operand"] == _demo_r
assert payload["sum"] == _demo_sum
assert payload["sum"] != 2 + 3
assert payload["proof_token"] == NB_FORMULA_SECRET
assert payload["formula"] == f"2+3+{_demo_r}=={_demo_sum}"
assert r_prov.execution.mode_used == ToolExecutionMode.DETERMINISTIC
print("  [PASS] Part 4 local proof — governed executor sum and proof_token match kernel baseline")

r_calc = run_tool(
    "calculate_result",
    {"operation": "multiply", "operand1": 8, "operand2": 9},
    "tc_calc",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
assert r_calc.status == ToolStatus.SUCCESS
assert r_calc.result is not None
assert r_calc.result["value"]["result"] == 72.0
assert r_calc.execution.mode_used == ToolExecutionMode.DETERMINISTIC

print("-- calculate_result divide-by-zero → structured TOOL_EXECUTION_ERROR --")
r_div0 = run_tool(
    "calculate_result",
    {"operation": "divide", "operand1": 10, "operand2": 0},
    "tc_div0",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
assert r_div0.status == ToolStatus.ERROR
assert r_div0.error is not None
assert r_div0.error.code == "TOOL_EXECUTION_ERROR"
assert r_div0.error.category == "tool_runtime"
div0_msg = r_div0.error.message
assert div0_msg is not None
assert "division by zero" in div0_msg
assert r_div0.execution.mode_used == ToolExecutionMode.DETERMINISTIC

r_echo = run_tool("risky_echo", {"msg": "hello"}, "tc_echo", risk_tier=RiskTier.HIGH, is_state_changing=True)
assert r_echo.status == ToolStatus.BLOCKED
assert r_echo.error is not None
assert r_echo.error.code == "POLICY_BLOCKED"
echo_msg = r_echo.error.message
assert echo_msg is not None
assert "manual review" in echo_msg

r_denied = run_tool("admin_reset", {}, "tc_denied", risk_tier=RiskTier.MEDIUM, is_state_changing=False)
assert r_denied.status == ToolStatus.BLOCKED
assert r_denied.error is not None
assert r_denied.error.code == "POLICY_BLOCKED"
denied_msg = r_denied.error.message
assert denied_msg is not None
assert "blocked by policy" in denied_msg.lower()

print("metrics counters:", metrics.counters)
assert metrics.counters == {
    "tool.call.total": 6,
    "tool.call.success": 3,
    "tool.call.failed": 1,
    "tool.call.blocked": 2,
}
print(
    "NB_FORMULA_SECRET is printed only for this local notebook proof. "
    "Do not expose equivalent production secrets in model-visible logs."
)
print("  demo proof_token (this kernel):", NB_FORMULA_SECRET)

print()
print("Contrast — same math, no policy / no executor / no metrics (not a supported agent path):")
print("  safe_add(2, 3) =>", safe_add(2, 3))
print("(Above, run_tool('safe_add', ...) went through policy + DeterministicToolExecutor + metrics.)")

before: allow LOW_RISK_ALLOWED
execute status: success
  error: None 
  mode_used: ToolExecutionMode.DETERMINISTIC
-- safe_add_proven: enterprise proof (sum = a + b + kernel random_operand) --
  random_operand (handler-only): 6649
  governed sum 2+3+6649 => 6654  |  plain 2+3 => 5 (wrong without tool)
  proof_token (this kernel): 7d89eebd51fb2e8b
before: allow RISK_WRITE_REQUIRES_DETERMINISTIC
execute status: success
  error: None 
  mode_used: ToolExecutionMode.DETERMINISTIC
  safe_add_proven JSON: {'operand_a': 2, 'operand_b': 3, 'random_operand': 6649, 'sum': 6654, 'proof_token': '7d89eebd51fb2e8b', 'formula': '2+3+6649==6654'}
  [PASS] Part 4 local proof — governed executor sum and proof_token match kernel baseline
before: allow LOW_RISK_ALLOWED
execute status: success
  error: None 
  mode_used: ToolExecutionMode.DETERMINISTIC
-- calculate_result divide-by-zero → structured TOOL_EXECUTION_ERROR --
before: allow LOW_RISK_ALLOWED
execute status: error
  error: TOOL_EXECUTION_ERROR d

## Part 5 — `select_execution_mode` (capability + policy)

**Story.** Even when policy **allows** a call, the product still chooses **how** it runs. Capability maps
describe the adapter (reliability, structured output support, etc.). `select_execution_mode` merges
**policy** (`enforced_mode`, risk tier, state-changing) with **capability** to pick
`deterministic` vs `provider_native`.

**Your task.** Edit **`CAPABILITY_VARIANTS`**: each entry’s `"kwargs"` is passed to `ProviderCapabilityMap`.
Compare the printed modes for the **same** `PolicyDecision.ALLOW` but different synthetic tool calls.

**Reading stdout.** **HIGH + state-changing** should stay **`deterministic`** even when the “weak” map
would otherwise prefer the provider — safety wins. For the default values here, **LOW** stays
**`provider_native`** in both capability maps; **HIGH + state-changing** remains **`deterministic`**
in both maps.

**Policy override:** when policy sets `enforced_mode=DETERMINISTIC`, mode selection honors it even for
low-risk calls — the optional assertion at the bottom of the cell demonstrates that.

In [6]:
from src.runtime.capability_map import ProviderCapabilityMap
from src.runtime.mode_selector import select_execution_mode
from src.schemas.tool_io import PolicyAction, PolicyDecision, RiskTier, ToolCallContext, ToolExecutionMode

CAPABILITY_VARIANTS = [
    {"label": "weak_capabilities", "kwargs": {"provider_id": "demo", "reliability_score": 5}},
    {
        "label": "strong_capabilities",
        "kwargs": {
            "provider_id": "demo",
            "supports_function_calling": True,
            "supports_structured_output": True,
            "reliability_score": 5,
        },
    },
]

allow = PolicyDecision(
    schema_version="1.0",
    decision=PolicyAction.ALLOW,
    reason_code="LOW_RISK_ALLOWED",
    message="ok",
    enforced_mode=None,
)

low = ToolCallContext(
    schema_version="1.0",
    call_id="m1",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="safe_add",
    arguments={},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
)
high = ToolCallContext(
    schema_version="1.0",
    call_id="m2",
    session_id="nb_sess",
    run_id="nb_run",
    job_id="nb_job",
    task_id="nb_task",
    agent_id="nb_agent",
    provider_id="demo",
    tool_name="risky_echo",
    arguments={"msg": "x"},
    tenant_id="tenant_nb",
    risk_tier=RiskTier.HIGH,
    is_state_changing=True,
)

print("Same tool intents; only the capability map changes:\n")
mode_results: dict[tuple[str, str], ToolExecutionMode] = {}
for variant in CAPABILITY_VARIANTS:
    caps = ProviderCapabilityMap(**variant["kwargs"])
    mode_results[(variant["label"], "low")] = select_execution_mode(low, caps, allow)
    mode_results[(variant["label"], "high")] = select_execution_mode(high, caps, allow)
    print(variant["label"], "low ->", mode_results[(variant["label"], "low")].value)
    print(variant["label"], "high ->", mode_results[(variant["label"], "high")].value)

assert mode_results[("weak_capabilities", "low")] == ToolExecutionMode.PROVIDER_NATIVE
assert mode_results[("weak_capabilities", "high")] == ToolExecutionMode.DETERMINISTIC
assert mode_results[("strong_capabilities", "low")] == ToolExecutionMode.PROVIDER_NATIVE
assert mode_results[("strong_capabilities", "high")] == ToolExecutionMode.DETERMINISTIC

forced = PolicyDecision(
    schema_version="1.0",
    decision=PolicyAction.ALLOW,
    reason_code="FORCED",
    message="forced",
    enforced_mode=ToolExecutionMode.DETERMINISTIC,
)
assert (
    select_execution_mode(low, ProviderCapabilityMap(provider_id="demo"), forced)
    == ToolExecutionMode.DETERMINISTIC
)

Same tool intents; only the capability map changes:

weak_capabilities low -> provider_native
weak_capabilities high -> deterministic
strong_capabilities low -> provider_native
strong_capabilities high -> deterministic


## Part 6 — Ingress gate chain (pre-model guard rails)

**Story.** **Ingress** answers: “Should this *text* become a billable model turn?” It runs **before**
the orchestrator. Custom rules, classifiers, and profile defaults all collapse into an ordered gate
chain with explicit **`gate_id`** and **`reason_code`** — ideal for SOC-style reviews.

**Your task.** Edit **`INGRESS_OVERLAY`** (profile, classifier mode, custom rules). Use
**`evaluate_prompt`** to send benign vs sensitive sample strings.

**Reading stdout.** The code runs **two** prompts back-to-back: a **benign** string (should **allow**)
and a **sensitive** string containing `SECRET_KEY` (should **deny** with your custom rule). That is the
simplest **with vs without** story for ingress: same chain, different user text, opposite outcomes —
and **no** model spend on the denied line.

**tutorial_09 reuse.** The same `chain` object is reused in **tutorial_09** when an API key is present.

In [7]:
from src.policies.ingress_gates import (
    IngressDecision,
    IngressGateChain,
    IngressTurnContext,
    build_ingress_gate_chain_from_overlay,
)
from src.policies.ingress_profiles import resolve_ingress_profile_settings
from src.schemas.tool_io import PolicyAction


def evaluate_prompt(
    chain: IngressGateChain,
    prompt: str,
    session_id: str = "nb-ingress",
) -> IngressDecision:
    ctx = IngressTurnContext(
        tenant_id="tenant_nb",
        session_id=session_id,
        correlation_id="corr-" + session_id,
        transport="notebook",
        user_input=prompt,
    )
    decision = chain.evaluate(ctx)
    msg = decision.message or ""
    print(decision.decision.value, decision.gate_id, decision.reason_code, "|", msg[:100])
    return decision


INGRESS_OVERLAY = {
    "ingress_profile": "baseline",
    "ingress_classifier_mode": "off",
    "ingress_custom_rules": [
        {
            "rule_id": "nb-block-secret",
            "action": "deny",
            "match_type": "contains_any",
            "patterns": ["SECRET_KEY", "BEGIN PRIVATE KEY"],
            "reason_code": "NB_SECRET_PATTERN",
            "message": "Blocked in notebook demo.",
        },
    ],
}

res = resolve_ingress_profile_settings(INGRESS_OVERLAY)
print("resolved profile:", res.profile_name, "custom rules:", len(res.custom_rules))

chain = build_ingress_gate_chain_from_overlay(INGRESS_OVERLAY)
d_ok = evaluate_prompt(chain, "hello world")
d_secret = evaluate_prompt(chain, "paste SECRET_KEY=abc here")

assert d_ok.decision == PolicyAction.ALLOW
assert d_ok.gate_id == "ingress-gate-chain"
assert d_ok.reason_code == "INGRESS_ALLOW_DEFAULT"

assert d_secret.decision == PolicyAction.DENY
assert d_secret.gate_id == "ingress-custom-rules"
assert d_secret.reason_code == "NB_SECRET_PATTERN"
assert d_secret.message == "Blocked in notebook demo."

assert res.profile_name == "baseline"
assert len(res.custom_rules) == 1
assert res.custom_rules[0].reason_code == "NB_SECRET_PATTERN"

print("PASS ingress deny on secret pattern")

resolved profile: baseline custom rules: 1
allow ingress-gate-chain INGRESS_ALLOW_DEFAULT | Turn allowed by ingress gate chain.
deny ingress-custom-rules NB_SECRET_PATTERN | Blocked in notebook demo.
PASS ingress deny on secret pattern


## Checkpoint — before Part 7 (orchestrator)

If anything below **fails**, use **Run All Above** from the next cell, or restart the kernel and run
from the **bootstrap** cell through **Part 6** without skipping.

**You should have seen:** Part 2 **escalate** on `s2`, Part 3 **`deny`** on `admin_reset` with overlay,
Part 4 **`success`** and one **`blocked`/`error`** line for divide-by-zero, Part 6 **`PASS ingress deny`**.

In [8]:
_missing = []
for _name in (
    "policy_overlay",
    "registry",
    "executor",
    "metrics",
    "chain",
    "evaluate_prompt",
    "NB_FORMULA_SECRET",
    "risk_cfg",
):
    if _name not in globals():
        _missing.append(_name)
if _missing:
    raise RuntimeError(
        "Checkpoint failed — re-run notebook from bootstrap through Part 6. Missing: " + ", ".join(_missing)
    )

from src.policies.ingress_gates import IngressGateChain
from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolRegistry

assert isinstance(registry, ToolRegistry)
assert registry.resolve("calculate_result") is not None
assert registry.resolve("safe_add_proven") is not None
assert isinstance(executor, DeterministicToolExecutor)
assert isinstance(chain, IngressGateChain)
assert callable(evaluate_prompt)
assert isinstance(NB_FORMULA_SECRET, str) and len(NB_FORMULA_SECRET) == 16

print("CHECKPOINT OK — continue to Part 7 (stub orchestrator). Optional live: tutorial_09.")

CHECKPOINT OK — continue to Part 7 (stub orchestrator). Optional live: tutorial_09.


## Part 7 — One-turn orchestrator (stub stream; no API key)

**Story.** The **orchestrator** ties the runtime adapter, policy, and executor into one **async event
stream** (`tool_progress`, `tool_intent`, `output_delta`, `run_complete`). For tests and notebooks,
`OpenAIAgentsRuntimeAdapter` supports **`planned_tool_call`**: a synthetic tool intent without calling
OpenAI. That lets you see **policy + deterministic execution + submit_tool_results** end-to-end.

**Your task.** Edit **`planned_tool_call`** (`tool_name`, `arguments`, `risk_tier`, `is_state_changing`)
so it matches a **registered** tool from Part 4. The default uses **`calculate_result`** × **8×9** with
**`risk_tier: medium`** and **`is_state_changing: true`** so you see **queued → running → completed**
without an API key.

**Important with your Part 1 config:** `USER_RISK` sets **`escalate_risk_tiers: ["high"]`**. A synthetic
intent marked **`high`** is **escalated → blocked** (`POLICY_BLOCKED`) — same as Part 2’s `s2` line. That is
correct policy behaviour, not a broken orchestrator. The cell runs a **second** planned call with
**`high`** to show that contrast after the completed **medium** path.

**Reading stdout (first run).** **queued → running → completed**, then **`output_delta`** /
**`run_complete`**. **Second run:** **queued → failed** with **`POLICY_BLOCKED`** — ties Part 2 to the stream.

**Note:** `RUN_COMPLETE` with `status=completed` can still occur when a tool result is **blocked** — the
orchestrator finished the turn by returning a typed blocked envelope, not by running the handler.

**With vs without OpenAI:** this part is **without** billing — `planned_tool_call` injects a tool
intent. **tutorial_09** (optional, API key) reuses the same **`planned_tool_call`** mechanism for **§2–§4**
governed proofs, plus ingress and raw Agents SDK contrasts; **§5** (off by default) is model-driven
diagnostic only.

In [9]:
from src.core.orchestrator import Orchestrator
from src.runtime.openai_agents_runtime import OpenAIAgentsRuntimeAdapter
from src.schemas.events import RuntimeEventType
from src.schemas.tool_io import ToolExecutionMode, ToolStatus


class CapturingOpenAIAgentsRuntimeAdapter(OpenAIAgentsRuntimeAdapter):
    def __init__(self) -> None:
        super().__init__()
        self.submitted_results: list = []

    async def submit_tool_results(self, session_id, run_id, tool_results):
        self.submitted_results.extend(tool_results)
        async for event in super().submit_tool_results(
            session_id=session_id,
            run_id=run_id,
            tool_results=tool_results,
        ):
            yield event


orch_adapter = CapturingOpenAIAgentsRuntimeAdapter()
orch = Orchestrator(
    runtime_adapter=orch_adapter,
    policy_middleware=policy_overlay,
    tool_executor=DeterministicToolExecutor(registry=registry, policy=policy_overlay, metrics=metrics),
)

ctx = {
    "run_id": "nb_orch_run",
    "job_id": "nb_orch_job",
    "task_id": "nb_orch_task",
    "agent_id": "nb_orch_agent",
    "planned_tool_call": {
        "call_id": "tc_orch_1",
        "tool_name": "calculate_result",
        "arguments": {"operation": "multiply", "operand1": 8, "operand2": 9},
        "risk_tier": "medium",
        "is_state_changing": True,
    },
}


def _progress_states(event_pairs: list) -> list[str]:
    states: list[str] = []
    for etype, payload in event_pairs:
        if etype == RuntimeEventType.TOOL_PROGRESS.value and isinstance(payload, dict):
            st = payload.get("state")
            if isinstance(st, str):
                states.append(st)
    return states


async def _run_planned(ctx: dict, *, label: str) -> list:
    print(f"\n-- {label} --")
    out: list = []
    async for ev in orch.run_turn("sess_nb", "run tool", ctx):
        out.append((ev.event_type.value, ev.payload))
        print(ev.event_type.value, ev.payload)
    return out


try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    events_ok = asyncio.run(_run_planned(ctx, label="MEDIUM + state-changing (expect completed)"))
else:
    try:
        import nest_asyncio
        nest_asyncio.apply()
        events_ok = loop.run_until_complete(_run_planned(ctx, label="MEDIUM + state-changing (expect completed)"))
    except ImportError:
        print("Install nest-asyncio for Jupyter: pip install nest-asyncio")
        raise

assert _progress_states(events_ok) == ["queued", "running", "completed"]

out_delta = next(
    payload for etype, payload in events_ok if etype == RuntimeEventType.OUTPUT_DELTA.value
)
assert "calculate_result" in out_delta["text"]
assert "tc_orch_1" in out_delta["text"]
assert "success" in out_delta["text"]
assert "72.0" in out_delta["text"]

run_complete = next(
    payload for etype, payload in events_ok if etype == RuntimeEventType.RUN_COMPLETE.value
)
assert run_complete["status"] == "completed"
assert run_complete["tool_results_count"] == 1
assert "calculate_result" in run_complete["tool_results_summary"]
assert "72.0" in run_complete["tool_results_summary"]
assert run_complete["provider_id"] == "openai"

assert len(orch_adapter.submitted_results) == 1
submitted_ok = orch_adapter.submitted_results[0]
assert submitted_ok.status == ToolStatus.SUCCESS
assert submitted_ok.result is not None
assert submitted_ok.result["value"]["result"] == 72.0
assert submitted_ok.execution.mode_used == ToolExecutionMode.DETERMINISTIC

print("PASS orchestrator stream — deterministic completed path")

ctx_high = dict(ctx)
ctx_high["planned_tool_call"] = {
    **ctx["planned_tool_call"],
    "call_id": "tc_orch_high_blocked",
    "risk_tier": "high",
}

try:
    loop = asyncio.get_running_loop()
except RuntimeError:
    events_blk = asyncio.run(_run_planned(ctx_high, label="HIGH risk (expect POLICY_BLOCKED — Part 1 escalate)"))
else:
    import nest_asyncio
    nest_asyncio.apply()
    events_blk = loop.run_until_complete(
        _run_planned(ctx_high, label="HIGH risk (expect POLICY_BLOCKED — Part 1 escalate)")
    )

assert _progress_states(events_blk) == ["queued", "failed"]

failed_progress = [
    payload for etype, payload in events_blk if etype == RuntimeEventType.TOOL_PROGRESS.value
][-1]
assert failed_progress["tool_status"] == "blocked"
assert failed_progress["error_code"] == "POLICY_BLOCKED"

blocked_delta = next(
    payload for etype, payload in events_blk if etype == RuntimeEventType.OUTPUT_DELTA.value
)
assert "blocked" in blocked_delta["text"]
assert "manual review" in blocked_delta["text"]

blocked_complete = next(
    payload for etype, payload in events_blk if etype == RuntimeEventType.RUN_COMPLETE.value
)
assert blocked_complete["status"] == "completed"
assert blocked_complete["tool_results_count"] == 1
assert "POLICY_BLOCKED" not in blocked_complete.get("status", "")

assert len(orch_adapter.submitted_results) == 2
submitted_blk = orch_adapter.submitted_results[1]
assert submitted_blk.status == ToolStatus.BLOCKED
assert submitted_blk.error.code == "POLICY_BLOCKED"

print("PASS orchestrator stream — HIGH intent blocked by policy (consistent with Part 2)")


-- MEDIUM + state-changing (expect completed) --
tool_progress {'call_id': 'tc_orch_1', 'tool_name': 'calculate_result', 'state': 'queued', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_orch_1', 'tool_name': 'calculate_result', 'state': 'running', 'tool_status': '', 'error_code': '', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
tool_progress {'call_id': 'tc_orch_1', 'tool_name': 'calculate_result', 'state': 'completed', 'tool_status': 'success', 'error_code': 'None', 'job_id': '', 'lease_token': '', 'lease_expires_at_epoch': '', 'claim_attempt': ''}
output_delta {'text': "- calculate_result (tc_orch_1): success → {'operation': 'multiply', 'operand1': 8.0, 'operand2': 9.0, 'result': 72.0}"}
run_complete {'status': 'completed', 'tool_results_count': 1, 'tool_results_summary': "- calculate_result (tc_orch_1): success → {'operation': 'multiply', '

## Summary — takeaways for integrators

| Part | Story in one line | What you edited | What to notice in stdout |
|------|-------------------|-----------------|---------------------------|
| 1–2 | Global risk defaults + synthetic intents | `USER_RISK`, `SCENARIOS` | Two passes: **your** rules vs **relaxed** rules |
| 3 | Tenant overlay merges on `tenant_id` | `USER_OVERLAY` | **Global-only** vs **with overlay** lines |
| 4 | Deterministic handlers are the trust boundary | `USER_TOOLS`, `run_tool` | **`safe_add_proven` JSON** (`random_operand`, `sum`, `proof_token`); plain a+b ≠ sum |
| 5 | Capability + policy choose execution mode | `CAPABILITY_VARIANTS` | LOW vs HIGH routing |
| 6 | Ingress is pre-model guard rails | `INGRESS_OVERLAY`, prompts | `gate_id`, ingress `reason_code` |
| 7 | Orchestrator stream without OpenAI | `planned_tool_call` | **MEDIUM** → **completed**; **HIGH** → **POLICY_BLOCKED** (matches Part 1) |

**Optional live contrasts:** **`tutorial_09_governed_execution_live.ipynb`** (after this notebook).

This notebook proves the **local governed execution path** (policy, deterministic tools, ingress,
stub orchestrator) — not raw SDK comparison or model choice. Tutorial 09 covers optional live contrasts.

Canonical ordering for **production** HTTP/SSE paths: `docs/architecture/governed-execution-pipeline.md`.
Customer-facing overlay keys and API behaviour: `docs/api/customer-api-integration-guide.md` and
`docs/strategy/customer-self-serve-governance-journey.md`.

## Notebook navigation

| If you want… | Open |
|---|---|
| Previous / next in learning path | See `notebooks/README.md` index |
| Fast module smoke after a code change | `check_01` … `check_04` |
| Ingress or tool boundary proofs | `edge_01`, `edge_02` |
| Local governance lab (no API key) | `tutorial_08_governed_execution_sandbox.ipynb` |
| Live governance contrasts (optional API key) | `tutorial_09_governed_execution_live.ipynb` |
| Evaluator time-boxed paths | `notebooks/EVALUATOR_GUIDE.md` |

**Regenerate notebooks:** edit this build script, then `python notebooks/build_tutorials.py` (do not hand-edit `.ipynb` JSON).